### Get zotero item html attachments

Zotero DB html meta info extraction done by load_zotero_data.ipynb

In [13]:
import pandas as pd
from icecream import ic
import pathlib as pl
from pyzotero import zotero
import sys

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw

In [14]:
def separator_page_metainfo(attachments):
    """Standardize merged output separator page meta information names"""
    info_renames = dict(Title='parentTitle',Author='parentFirstCreator',Source='parentVenue', Date='parentDate')
    return attachments[info_renames.values()].rename(info_renames, axis=1) # a dataframe, colname is metainfo name

In [15]:
zotinfo = rfw.load_pickle_data(rfw.extractedZoteroEntriesFNm) # all attachments from load_zotero_data.ipynb

# select which attachments to merge
merge_attachments = zotinfo['attachment_files'].query('contentType == "text/html"')
merge_attachments = merge_attachments[merge_attachments['parentCollections'].apply(lambda x: 'Hot Takes' in x)]

Reading from C:\Users\scott\OneDrive\share\ref\refwrangle\dat\zotero_entries.pkl...


In [17]:
def add_fail(fail_list, fileInfo, failInfo):
    fail_list.append(failInfo | fileInfo)

min_pdf_bytes = 100e3
html_failures = []
for ix, attachment in merge_attachments.iterrows():
    fileInfo = dict(citekey=attachment.parentCitekey)
    html_file = attachment.file_fullpath
    try:
        pdf_file = rfw.html2pdf_cachedir / f"{html_file.stem}.pdf"
    except:
        print(f'Skipping bad html_file {fileInfo["citekey"]}')
        add_fail(html_failures, fileInfo, dict(fail_type='bad_html_file'))
        continue
    
    if pdf_file.exists():
        print(f'pdf file exists, so skipping: {fileInfo["citekey"]}')
        continue
    
    fileInfo = fileInfo | dict(html_file=html_file.name, pdf_file=pdf_file.name)
    if (not pdf_file.exists() or (nbytes := pdf_file.stat().st_size) < min_pdf_bytes 
        or html_file.stat().st_mtime > pdf_file.stat().st_mtime):
        print(f"{html_file.name} --> {pdf_file.name}")
        try:
            rfw.convert_html_to_pdf_subproc(html_file, pdf_file)
            if nbytes < min_pdf_bytes:
                print(f"{pdf_file.name} {nbytes=} less than {min_pdf_bytes} on {html_file.name}")
            add_fail(html_failures, fileInfo, dict(fail_type='tooshort', nbyte=nbytes))
        except:
            print(f"Failed on {html_file.name}")
            add_fail(html_failures, fileInfo, dict(fail_type='exception'))
    else:
        print(f"Skipping {html_file.name}, PDF is up to date.")


print(f"\nDone: {(nFails := len(html_failures))} conversion falures")
if nFails > 0:
    print("Failures:")
    display(pd.DataFrame(html_failures))

pdf file exists, so skipping: Cousens24voterEngageHistoryPost
pdf file exists, so skipping: Otte24whyVotersNoVote
pdf file exists, so skipping: Brownstein24trumpBetrayRural
pdf file exists, so skipping: Marantz24demsPartyOfElites
pdf file exists, so skipping: DavisWSUStudyPresidential
pdf file exists, so skipping: Johnson24identityPoliticsIsntWhy
pdf file exists, so skipping: Blueprint24pollPosHarrisCase
pdf file exists, so skipping: FitzGerald24howBigTrumpWin
pdf file exists, so skipping: Epstein24demBrightSpots
pdf file exists, so skipping: Dowd24demsMistakenIdentyPol
pdf file exists, so skipping: Kristof24demsNeedWrkingClssActLikeIt
pdf file exists, so skipping: McArdle24demsStopAbortCmpgn
pdf file exists, so skipping: Bradner24trumpVoterShifts
pdf file exists, so skipping: Sanders24demoGroups5Voted
pdf file exists, so skipping: Stewart24divideDemsWorkingClass
pdf file exists, so skipping: Nover24disaffectDemsVoteSwap
pdf file exists, so skipping: Kalefa24whiteGrievTrumpHispanic
pdf

In [18]:
# metainfo = separator_page_metainfo(merge_attachments)

# # Do the merge
# pdfs_info = []
# for ix, fullpath in merge_attachments.file_fullpath.items():
#     pdfs_info.append({'file':str(fullpath), 'metainfo':metainfo.loc[ix].to_dict()})

# rfw.merge_pdfs_with_structure(pdfs_info, "merged_articles.pdf")
# print('Done.')